# Observability with Phoenix Tracing

In [21]:
from dotenv import load_dotenv
import os
load_dotenv()

# Verify it loaded
print("API key set:", "Yes" if os.getenv('NVIDIA_API_KEY') else "No")

API key set: Yes


In [22]:
# uncomment to install in your own environment
# !pip install "nvidia-nat[phoenix]"

## Start Phoenix

In [23]:
import subprocess
import time
import sys

# Start Phoenix 
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Read initial output
print("Starting Phoenix...")
start_time = time.time()
while time.time() - start_time < 3:
    line = phoenix_process.stdout.readline()
    if line:
        print(line.strip())

print("\n✅ Phoenix is running silently in the background!")

Starting Phoenix...
🏃‍♀️‍➡️ Running migrations on the database.

✅ Phoenix is running silently in the background!


## Access Phoenix UI
The following codeblock is just some quick html to get the URL for your Phoenix server: 

In [24]:
from IPython.display import HTML, display

html = """
<div style="padding: 15px; background-color: #e8f4f8; border-radius: 8px; margin: 10px 0;">
    <h4 style="margin: 0 0 10px 0;">🔍 Phoenix Observability Dashboard</h4>
    <p>Click the button to open Phoenix (it will auto-detect the correct URL):</p>
    <button onclick="
        var url = window.location.origin.replace(/p\\d+/, 'p6006');
        window.open(url, '_blank');
    " style="padding: 10px 20px; background-color: #0066cc; color: white; border: none; border-radius: 4px; cursor: pointer; font-size: 14px;">
        🚀 Open Phoenix UI
    </button>
    <p style="margin-top: 10px; font-size: 12px; color: #666;">
        If the link doesn't work, manually replace the port in your browser URL from p8888 to p6006
    </p>
</div>
"""

display(HTML(html))

## Setup Climate Analyzer

In [25]:
%%capture
# Install the package so it's ready for all subsequent cells
!cd climate_analyzer && pip install -e . && cd ..

## Phoenix Configuration

In [ ]:
# %load climate_analyzer/src/climate_analyzer/configs/config.yml

general:                                              # Global settings for the workflow
  telemetry:                                          # Monitoring and metrics collection
    tracing:                                          # Distributed tracing for debugging/observability
      phoenix:                                        # Phoenix-specific tracing configuration
        _type: phoenix                                # Phoenix observability provider
        endpoint: http://localhost:6006/v1/traces     # Where to send trace data
        project: climate_analyzer_baseline            # Project name in Phoenix UI

llms:
  climate_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: $NVIDIA_BASE_URL 
    api_key: $NVIDIA_API_KEY
    temperature: 0.0
    top_p: 0.95
    max_tokens: 2048

functions:
  list_countries:
    _type: climate_analyzer/list_countries
    description: "List all available countries in the dataset"
    
  calculate_statistics:
    _type: climate_analyzer/calculate_statistics
    description: "Calculate temperature statistics globally or for a specific country"
  
  filter_by_country:
    _type: climate_analyzer/filter_by_country
    description: "Get information about climate data for a specific country"
  
  find_extreme_years:
    _type: climate_analyzer/find_extreme_years
    description: "Find the warmest or coldest years in the dataset"
  
  create_visualization:
    _type: climate_analyzer/create_visualization
    description: "Create visualizations including automatic top 5 countries by warming trend (country_comparison plot)"

workflow:
  _type: react_agent
  tool_names:
    - list_countries
    - calculate_statistics
    - filter_by_country
    - find_extreme_years
    - create_visualization
  llm_name: climate_llm
  verbose: true
  max_iterations: 5
  parse_agent_response_max_retries: 2
  max_tool_calls: 30


## Run Test Queries
These queries are designed to expose different behaviors—some will work well, others will reveal inefficiencies.

In [27]:
queries = [
    "What is the warming rate for Canada?",
    "What is the second coldest year in the dataset?",
    "Which country has the most weather stations in our data?"
]

In [28]:
# run each query in the list 
for query in queries:
    print(f"\nRunning: {query}")
    !nat run --config_file climate_analyzer/src/climate_analyzer/configs/config.yml --input "$query"
    print("-" * 60)


Running: What is the warming rate for Canada?
2026-04-18 05:53:21 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'climate_analyzer/src/climate_analyzer/configs/config.yml'
2026-04-18 05:53:21 - INFO     - phoenix.config:1750 - 📋 Ensuring phoenix working directory: /home/jovyan/.phoenix
2026-04-18 05:53:21 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_764dbfd9-2d68-401b-949b-0fafe351300d initialized
2026-04-18 05:53:25 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 05:53:25 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appende

2026-04-18 05:53:58 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'climate_analyzer/src/climate_analyzer/configs/config.yml'
2026-04-18 05:53:58 - INFO     - phoenix.config:1750 - 📋 Ensuring phoenix working directory: /home/jovyan/.phoenix
2026-04-18 05:53:58 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_ae650f89-f768-4095-97b2-c8584fb39063 initialized
2026-04-18 05:54:03 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 05:54:03 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(

Configu

2026-04-18 05:56:15 - INFO     - nat.agent.react_agent.agent:193 - 
------------------------------
[AGENT]
Agent input: Which country has the most weather stations in our data?
Agent's thoughts: 
Thought: I now have the information for Comoros. The number of stations in Comoros is less than in Argentina, Austria, and Brazil. I will continue this process for the remaining countries to find the one with the most weather stations. Action: filter_by_country Action Input: {"country_name": "France"}
------------------------------
2026-04-18 05:56:15 - INFO     - nat.agent.base:221 - 
------------------------------
[AGENT]
Calling tools: filter_by_country
Tool's input: {'country_name': 'France'}
Tool's response: 
{
  "country": "France",
  "stations": 3,
  "records": 187,
  "years": "1950-2025",
  "stations_list": [
    {
      "station_id": "FRE00000745",
      "name": "DIJON"
    },
    {
      "station_id": "FRE00104907",
      "name": "MONTELIMAR"
    },
    {
      "station_id": "FRE0010

2026-04-18 05:56:34 - INFO     - nat.agent.react_agent.agent:193 - 
------------------------------
[AGENT]
Agent input: Invalid Format: Missing 'Action:' after 'Thought:'

Agent's thoughts: 

------------------------------
2026-04-18 05:56:34 - INFO     - nat.agent.react_agent.agent:237 - [AGENT] Retrying ReAct Agent, including output parsing Observation
2026-04-18 05:56:34 - INFO     - nat.utils.exception_handlers.automatic_retries:159 - Retrying on exception [429] Too Many Requests
{'status': 429, 'title': 'Too Many Requests'} with matched message [429] too many requests
{'status': 429, 'title': 'too many requests'}
2026-04-18 05:56:34 - INFO     - nat.agent.react_agent.agent:193 - 
------------------------------
[AGENT]
Agent input: Invalid Format: Missing 'Action:' after 'Thought:'

Agent's thoughts: 

------------------------------
2026-04-18 05:56:34 - WARNING  - nat.agent.react_agent.agent:226 - [AGENT] Failed to parse agent output after 2 attempts, consider enabling or increasi

------------------------------------------------------------


## Fix: Add Station Tool
Update your config to include a new `station_statistics` tool by running the following line of code: 

In [ ]:
%load climate_analyzer/src/climate_analyzer/configs/config_updated.yml

```yaml
functions:
  # New tool to address observed churn
  station_statistics:
    _type: climate_analyzer/station_statistics
    description: "Get statistics on climate stations used in the data"

workflow:
  _type: react_agent
  tool_names:
    - list_countries
    - calculate_statistics
    - filter_by_country
    - find_extreme_years
    - create_visualization
    # Our new tool
    - station_statistics
```

### Test the Fix
Now run the same queries with the updated config:

In [29]:
for query in queries:
    print(f"\nRunning: {query}")
    !nat run --config_file climate_analyzer/src/climate_analyzer/configs/config_updated.yml --input "$query"
    print("-" * 60)


Running: What is the warming rate for Canada?
2026-04-18 05:57:00 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'climate_analyzer/src/climate_analyzer/configs/config_updated.yml'
2026-04-18 05:57:00 - INFO     - phoenix.config:1750 - 📋 Ensuring phoenix working directory: /home/jovyan/.phoenix
2026-04-18 05:57:00 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_1f2c0181-c282-4440-a5e7-71e092073a5b initialized
2026-04-18 05:57:04 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 05:57:04 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is

------------------------------------------------------------

Running: What is the second coldest year in the dataset?
2026-04-18 05:57:16 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'climate_analyzer/src/climate_analyzer/configs/config_updated.yml'
2026-04-18 05:57:16 - INFO     - phoenix.config:1750 - 📋 Ensuring phoenix working directory: /home/jovyan/.phoenix
2026-04-18 05:57:16 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_690ea07a-a39b-46b9-8a25-5b75c61a9677 initialized
2026-04-18 05:57:20 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
2026-04-18 05:57:20 - WARNING  - nat.builder.function_info:455 - Using provided input_schema for multi-argument function
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues.

## Clean Up

In [30]:
# Stop Phoenix server
phoenix_process.terminate()
phoenix_process.wait()
print("✅ Phoenix stopped")

✅ Phoenix stopped


In [31]:
# Uninstall workflow
!pip uninstall climate_analyzer -y

Found existing installation: climate_analyzer 0.1.0
Uninstalling climate_analyzer-0.1.0:
  Successfully uninstalled climate_analyzer-0.1.0
